# NITK-UsoundSim v2: runner-by-runner demo (phantom in, final image out)

The whole simulator runs **inside this notebook**, one section per runner, in pipeline order. Each section says what the runner **receives**, what it **does**, and its **output** (name, shape, units), then plots it. Nothing is loaded from saved results.

| Team | Runner | Folder | Output handed to the next stage |
|---|---|---|---|
| T1 Transducer | (probe) | `teams/T1_Transducer` | probe: element positions, centre frequency |
| T2 TX Beamforming | (transmit) | `teams/T2_TX_Beamforming` | transmit pulse, delays and apodization per element |
| T4 Tissue Interaction | 1 stage 1 | `teams/T4_Tissue_Interaction` | `scatterer_xyz` (N, 3) m, reflectivity |
| T3 Acoustic Propagation | 2 forward | `teams/T3_Acoustic_Propagation` | `incident_pressure` (N, T) |
| T4 Tissue Interaction | 3 stage 2 | `teams/T4_Tissue_Interaction` | `reflected_pressure` (N, T) |
| T3 Acoustic Propagation | 4 return | `teams/T3_Acoustic_Propagation` | `received_pressure` (128, T) |
| T5 RX Beamforming | 5 | `teams/T5_RX_Beamforming` | `beamformed_rf` (depth, lines) |
| T6 Image Reconstruction | 6 | `teams/T6_Image_Reconstruction` | `recon_rf` (depth, lines) conditioned RF |
| T7 B-mode | 7 | `teams/T7_B_mode` | `bmode` (depth, lines) in [0, 1] |
| T8 Post-Image Processing | 8 | `teams/T8_Post_Image_Processing` | `final` image (H, W) in [0, 1] |

**Any phantom works:** a built-in preset, a CSV of points, or your own image with a mapping file. Change it in the **settings cell** below.

# Run it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/darshilgmaniya/NITK-UsoundSim-v2/blob/main/demo_colab.ipynb)

**How to run:** `Runtime -> Run all` in Colab, or run the cells top to bottom on your computer.

1. **Step 0** (next cell) finds the project, using the first option that works: a project folder you are already in or have in `/content`; a project `.zip` in `/content` (it is unzipped); otherwise it downloads the project from GitHub (`git clone`).
2. The **settings cell** is the only cell you need to edit (phantom, probe, options).
3. The final image is shown near the end and saved as `final_image_demo.png`.

In [ ]:
# Step 0: find the project on this computer (VS Code / Jupyter). In Colab: upload the project .zip to /content.
import glob, os, subprocess, sys, zipfile
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
MARK = Path("integration") / "03_full_pipeline" / "run_pipeline.py"

def find_project():
    starts = [Path.cwd()]
    nb = globals().get("__vsc_ipynb_file__")                  # VS Code gives the path of this notebook
    if nb:
        starts.insert(0, Path(nb).resolve().parent)
    for s in starts:
        for d in [s, *s.parents]:                              # this folder and every parent
            if (d / MARK).exists():
                return d
    for s in starts + [BASE]:                                  # or a project folder just below (up to 2 levels)
        for pat in ("*", "*/*"):
            for d in sorted(s.glob(pat)):
                if d.is_dir() and (d / MARK).exists():
                    return d
    return None

project = find_project()
if project is None and glob.glob(str(BASE / "*.zip")):         # Colab: an uploaded project zip
    zip_path = sorted(glob.glob(str(BASE / "*.zip")))[0]
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BASE)
    print("unzipped:", zip_path)
    project = find_project()
assert project is not None, ("Project folder not found. In VS Code use File > Open Folder... and open the "
                             "NITK-UsoundSim-v2 folder (the one that contains 'teams' and 'integration'), then re-run.")
ROOT = project.resolve()
print("project:", ROOT)

# the only extra packages that may be missing
try:
    import cv2, yaml, PIL                                      # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pyyaml", "pillow"], check=True)


## Setup: import every module (`_paths.py` makes them importable without installing)

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

PIPE = ROOT / "integration" / "03_full_pipeline"
sys.path.insert(0, str(PIPE))
import _paths                                                          # noqa: F401  (adds all module folders to sys.path)

from usoundsim_relay import RelayConfig, run_relay                      # runners 1-4
import receiver_beamforming as rb                                       # runner 5
from receiver_beamforming import RxConfig
from image_reconstruction import image_reconstruction, ReconConfig       # runner 6
from bmode_formation import bmode_formation                             # runner 7
from post_processing_stage import scan_convert, postprocess, speckle_index, to_uint8   # runner 8
from usoundsim_full import demo_points
sys.path.insert(0, str(ROOT / 'teams' / 'T1_Transducer' / 'src'))                       # T1 code (plain modules)
sys.path.insert(0, str(ROOT / 'teams' / 'T2_TX_Beamforming' / 'Usound'))                 # T2 code (package 'src')
from registry import registry                                           # T1
from process import process_material                                    # T1
from src.transducer_interface import TransducerConfig                   # T2
from src.beamformer import build_tx_package, generate_element_pulses    # T2
from usoundsim_pipeline import PostConfig, run_pipeline                 # the same chain in one call (used at the end)
from usoundsim_pipeline.upload import show_upload_widget, collect_upload # live file upload

# some project modules force the non-interactive "Agg" backend when imported -> switch back so figures show in the notebook
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass
plt.rcParams["figure.dpi"] = 100
print("Python:", sys.version.split()[0], "| numpy", np.__version__)
print("project root:", ROOT)

## Settings: choose the phantom and the options (the only cell you edit)

| `SOURCE` | Meaning |
|---|---|
| `"points"` | nine wire targets (5 seconds) |
| `"sample_abdomen"` or `"sample_thyroid"` | the label-map images in `teams/T4_Tissue_Interaction/sample_data` (30 seconds) |
| a preset name such as `"abdomen"`, `"kidney"`, `"thyroid"`, `"breast"` | built-in phantom |
| `"my_points.csv"` | CSV with `x_mm,z_mm,amp` or `x_mm,z_mm,tissue` |
| `"phantom.png"` + `MAPPING = "map.json"` | your own image phantom |

`FIT_APERTURE = True` picks the pixel size so an image phantom is 95 % as wide as the array. Without it, a phantom wider than the array is clipped to the array width.
The default probe is the updated Acoustic Propagation config (128 elements, 5 MHz, pitch lambda/2 = 0.154 mm, aperture 19.6 mm). A pitch above lambda (for example 0.30 mm at 8 MHz) gives grating-lobe artefacts.

In [ ]:
SOURCE         = "sample_abdomen"      # see the table above
MAPPING        = None                  # mapping .json (needed for your own image phantom)
FIT_APERTURE   = True                  # image phantoms only
SPACING_MM     = None                  # pixel size of an image phantom [mm]; None = from the mapping file or FIT_APERTURE
MAX_SCATTERERS = 20000                 # more = finer speckle but slower

N_ELEMENTS, F0_MHZ, PITCH_MM = None, None, None     # None = the defaults of the Acoustic Propagation config

APOD, F_NUMBER = "hann", 1.0           # receive apodisation and F-number
DR             = 50.0                  # B-mode dynamic range [dB]
DESPECKLE      = "guided"              # "guided" or "none"

## Upload your own phantom while the notebook runs (optional)
Run the next cell: a **file dialog opens** on your computer (Colab: its upload dialog; if no dialog is possible, an Upload button appears). Select the phantom **and its mapping file together**:

| You upload | Needs |
|---|---|
| an image (`.png`, `.jpg`) | its mapping `.json` (select both at once) |
| a points file `.csv` (`x_mm,z_mm,amp` or `x_mm,z_mm,tissue`) | nothing else |
| an `(N, 3)` array `.npy` (`x_mm, z_mm, amp`) | nothing else |

The dialog blocks until you pick the files; then the cells below run on them. (*Run All* opens the dialog once, Cancel = default phantom.)
If you upload nothing, the `SOURCE` of the settings cell above is used.

In [ ]:
uploader = show_upload_widget()

In [ ]:
SAMPLES = ROOT / "teams" / "T4_Tissue_Interaction" / "sample_data"
up = collect_upload(uploader) if "uploader" in globals() else None      # files chosen in the Upload button, if any
if up:
    SOURCE, MAPPING = up.source, up.mapping
    print("using your uploaded phantom:", up)
src, mapping = SOURCE, MAPPING
if SOURCE == "points":
    src = demo_points()
elif SOURCE in ("sample_abdomen", "sample_thyroid"):
    name = SOURCE.split("_")[1]
    src, mapping = str(SAMPLES / f"phantom_{name}_labels.png"), str(SAMPLES / f"phantom_{name}_mapping.json")

kw = {}
if N_ELEMENTS: kw["n_elements"] = int(N_ELEMENTS)
if F0_MHZ:     kw["f0_hz"] = F0_MHZ * 1e6
if PITCH_MM:   kw["pitch_m"] = PITCH_MM * 1e-3
rc = RelayConfig(max_scatterers=MAX_SCATTERERS, also_ti_channel=False, **kw)
rx = RxConfig(apodization=APOD, f_number=F_NUMBER or None, dynamic_range_db=DR)
post = PostConfig(dynamic_range_db=DR, despeckle=DESPECKLE)

spacing = SPACING_MM
if FIT_APERTURE and isinstance(src, str) and src.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
    from PIL import Image
    width_px = Image.open(src).size[0]
    aperture_mm = (rc.n_elements - 1) * rc.pitch_m * 1e3
    spacing = 0.95 * aperture_mm / width_px
    print(f"FIT_APERTURE: image {width_px} px wide, aperture {aperture_mm:.1f} mm -> pixel size {spacing:.4f} mm")
print(f"phantom: {SOURCE} | probe: {rc.n_elements} elements, pitch {rc.pitch_m*1e3:.3f} mm, f0 {rc.f0_hz/1e6:g} MHz, "
      f"aperture {(rc.n_elements-1)*rc.pitch_m*1e3:.1f} mm")

---
# T1 Transducer: the probe
**Receives:** nothing from the phantom. **Does:** defines the array. T1's registry holds the piezo material and its allowed frequency range; the probe used in this run (set in the settings cell) must have its centre frequency inside that range.
**Output:** element positions `element_x` (128,) in metres and the centre frequency `f0`.

In [ ]:
registry.clear()
f_draw = process_material("PZT-5H", (1000.0, 12000.0), create_new=True)        # T1 code: material + allowed range (kHz)
print(f"T1 registry: {registry.list_materials()} | allowed range 1-12 MHz | this run f0 = {rc.f0_hz/1e6:g} MHz -> inside range: {1e6 <= rc.f0_hz <= 12e6}")
element_x = (np.arange(rc.n_elements) - (rc.n_elements - 1) / 2) * rc.pitch_m
fig, ax = plt.subplots(1, 1, figsize=(11, 1.5))
ax.scatter(element_x*1e3, np.zeros_like(element_x), marker="s", s=14); ax.set_yticks([])
ax.set(title=f"T1 output: {rc.n_elements} elements, pitch {rc.pitch_m*1e3:.3f} mm, aperture {(element_x[-1]-element_x[0])*1e3:.1f} mm, f0 {rc.f0_hz/1e6:g} MHz", xlabel="element x (mm)")
plt.tight_layout(); plt.show()

---
# T2 TX Beamforming: the transmit
**Receives:** the probe from T1. **Does:** per-element transmit delays, apodization weights and element pulses (`build_tx_package`), here for a focus at `TX_FOCUS_MM` depth.
**Output:** `tx_delays_s` (128,), `tx_weights` (128,), pulses (128, samples).
**Honest note:** the integrated T3 code transmits with all delays = 0 (every element fires together, an unfocused broadcast). The delays below are T2's focus plan; they are shown and checked, but not applied in this run.

In [ ]:
TX_FOCUS_MM = 20.0
tc = TransducerConfig("v2 probe", rc.n_elements, rc.pitch_m, 0.4*rc.f0_hz, 2.4*rc.f0_hz, rc.f0_hz)
beam = dict(focus_x_m=0.0, focus_z_m=TX_FOCUS_MM*1e-3, steering_angle_deg=0.0, sound_speed_m_s=rc.c0, apodization="hann", pulse_duration_sigma_cycles=1.0)
tx = build_tx_package(tc, beam)
ts, ps = generate_element_pulses(tx["tx_delays_s"], tx["tx_frequency_hz"], tx["tx_weights"])
print("T2 output: tx_delays_s", tx["tx_delays_s"].shape, f"({tx['tx_delays_s'].min()*1e6:.2f}..{tx['tx_delays_s'].max()*1e6:.2f} us) | tx_weights", tx["tx_weights"].shape, "| pulses", ps.shape)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
ax[0].plot(tx["element_x_m"]*1e3, tx["tx_delays_s"]*1e6, marker="o", ms=2); ax[0].set(title=f"T2: transmit delays (focus {TX_FOCUS_MM:g} mm)", xlabel="element x (mm)", ylabel="delay (us)")
ax[1].plot(tx["element_x_m"]*1e3, tx["tx_weights"]); ax[1].set(title="T2: apodization weights (hann)", xlabel="element x (mm)")
ax[2].imshow(ps, aspect="auto", cmap="seismic", extent=[ts[0]*1e6, ts[-1]*1e6, ps.shape[0], 0]); ax[2].set(title="T2: element pulses", xlabel="time (us)", ylabel="element")
plt.tight_layout(); plt.show()

---
# T4 + T3, runners 1 to 4: Tissue Interaction and Acoustic Propagation
**Receives:** the phantom and the probe settings. **Does:**
1. *Tissue Interaction, stage 1* turns the phantom into scatterers (position, reflectivity).
2. *Acoustic Propagation, forward*: the transmit pulse travels from the 128 elements to every scatterer, with delay and attenuation.
3. *Tissue Interaction, stage 2* turns the incident pressure at each scatterer into reflected pressure (impedance, speckle, shadows).
4. *Acoustic Propagation, return*: each echo travels back to the 128 elements and all echoes are summed.

**Output:** `received_pressure` (128 elements x T samples) with the real `t_axis` (it starts before 0: every later delay lookup must use this axis).

In [ ]:
t0 = time.time()
relay = run_relay(src, rc, mapping=mapping, spacing_mm=spacing, background="generic_soft", units="mm", verbose=False)
print(f"relay done in {time.time()-t0:.1f} s")
for k, v in relay.timings.items():
    print(f"  {k:28s} {v:6.1f} s")
xyz = relay.scatterers.xyz
print("scatterers:", xyz.shape, "| incident:", relay.incident.shape, "| reflected:", relay.reflected.shape,
      "| received_pressure:", relay.received_pressure.shape, f"| t_axis {relay.t_axis[0]*1e6:.2f} .. {relay.t_axis[-1]*1e6:.1f} us")

In [ ]:
amp = np.abs(relay.scatterers.amp); order = np.argsort(amp)
fig, ax = plt.subplots(1, 4, figsize=(18, 4.6), gridspec_kw=dict(width_ratios=[1.2, 1.2, 1, 1.3]))
ax[0].scatter(xyz[order, 0]*1e3, xyz[order, 2]*1e3, c=amp[order], s=2, cmap="gray_r")
ax[0].invert_yaxis(); ax[0].set_aspect("equal"); ax[0].set(title=f"T4 runner 1: {len(xyz)} scatterers", xlabel="x (mm)", ylabel="z (mm)")
ax[1].plot(relay.pulse_t*1e6, relay.pulse_p); ax[1].set(title=f"T3 transmit pulse ({rc.f0_hz/1e6:g} MHz)", xlabel="time (us)")
k = int(np.argmax(np.abs(relay.incident).max(axis=1)))
ax[2].plot(relay.t_axis*1e6, relay.incident[k], lw=0.8, label="T3 incident (runner 2)")
ax[2].plot(relay.t_axis*1e6, relay.reflected[k], lw=0.8, label="T4 reflected (runner 3)")
ax[2].set(title="strongest scatterer", xlabel="time (us)"); ax[2].legend(fontsize=8)
rp = relay.received_pressure; disp = np.sign(rp)*np.abs(rp)**0.4; lim = np.percentile(np.abs(disp), 99.5)
ax[3].imshow(disp.T, aspect="auto", cmap="gray", vmin=-lim, vmax=lim,
             extent=[0.5, rc.n_elements+0.5, relay.t_axis[-1]*1e6, relay.t_axis[0]*1e6])
ax[3].set(title=f"T3 runner 4: received_pressure {rp.shape}", xlabel="receive element", ylabel="time (us)")
plt.tight_layout(); plt.show()

---
# T5 RX Beamforming, runner 5: delay and sum
**Receives:** `received_pressure`, `t_axis`, element positions. **Does:** for every scan line and depth, computes the round-trip delay to each element, reads each channel with `np.interp(tau, t_axis, rf)` on the **real** time axis, applies a window and sums (dynamic receive focus, dynamic aperture set by the F-number).
**Output:** `beamformed_rf` (depth samples x scan lines) with `z_axis` and the scan-line positions `x_lines`.

In [ ]:
inp = rb.make_receiver_input(relay.received_pressure, relay.t_axis, relay.element_x, fs_hz=relay.rc.fs_hz, c0=relay.rc.c0,
                             f0_hz=relay.rc.f0_hz, pitch_m=relay.rc.pitch_m, scatterer_xyz=relay.scatterers.xyz, source="notebook")
t0 = time.time()
bf = rb.beamform(inp, rx)
beamformed_rf, z_axis, x_lines = bf.beamformed_rf, bf.z_axis, bf.x_lines
print(f"beamformed_rf: {beamformed_rf.shape} (depth, lines) | z_axis {z_axis[0]*1e3:.1f}-{z_axis[-1]*1e3:.1f} mm | "
      f"x_lines {x_lines[0]*1e3:.1f}..{x_lines[-1]*1e3:.1f} mm | {time.time()-t0:.1f} s")
for n in bf.notes:
    print("note:", n)
lim = np.percentile(np.abs(beamformed_rf), 99.9)
plt.figure(figsize=(5, 6)); plt.imshow(beamformed_rf, aspect="equal", cmap="gray", vmin=-lim, vmax=lim, extent=bf.extent_mm)
plt.xlabel("x (mm)"); plt.ylabel("z (mm)"); plt.title("T5 runner 5 output: beamformed_rf"); plt.show()

---
# T6 Image Reconstruction, runner 6
**Receives:** `beamformed_rf`. **Does:** removes the DC offset of every scan line and applies a zero-phase band-pass filter around the centre frequency (`image_reconstruction`). The depth axis is not changed.
**Output:** `recon_rf` (depth x lines), same shape as `beamformed_rf`; handed to runner 7 (B-mode).

In [ ]:
rec = image_reconstruction(beamformed_rf, z_axis, c=relay.rc.c0, f0_hz=relay.rc.f0_hz, config=ReconConfig())
recon_rf = rec.rf
print("recon_rf:", recon_rf.shape, "|", "; ".join(rec.notes))
li = int(np.argmin(np.abs(x_lines))); zi = int(np.argmax(np.abs(beamformed_rf[:, li]))); sl = slice(max(zi - 150, 0), zi + 150)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(z_axis[sl]*1e3, beamformed_rf[sl, li], lw=0.8, label="beamformed_rf (runner 5)"); ax[0].plot(z_axis[sl]*1e3, recon_rf[sl, li], lw=0.8, label="recon_rf (runner 6)")
ax[0].set(title="centre line, around the strongest echo", xlabel="depth (mm)"); ax[0].legend()
f = np.fft.rfftfreq(beamformed_rf.shape[0], d=1/rec.fs_depth_hz)/1e6
ax[1].plot(f, np.abs(np.fft.rfft(beamformed_rf[:, li])), lw=0.8, label="before"); ax[1].plot(f, np.abs(np.fft.rfft(recon_rf[:, li])), lw=0.8, label="after")
ax[1].set(title="spectrum of the centre line", xlabel="frequency (MHz)", xlim=(0, 15)); ax[1].legend()
plt.tight_layout(); plt.show()

---
# T7 B-mode Formation, runner 7
**Receives:** `recon_rf`. **Does:** envelope by the Hilbert transform along depth, `20*log10(envelope / max)`, clip to the dynamic range, map to [0, 1] (`bmode_formation`).
**Output:** `bmode` (depth x lines), 0 (at or below the dynamic range) to 1 (0 dB); the array handed to runner 8.

In [ ]:
envelope = np.abs(hilbert(recon_rf, axis=0))
bmode = bmode_formation(recon_rf, DR)
print("envelope:", envelope.shape, "| bmode:", bmode.shape, f"range [{bmode.min():.2f}, {bmode.max():.2f}]")
li = int(np.argmin(np.abs(x_lines))); zi = int(np.argmax(envelope[:, li])); sl = slice(max(zi - 300, 0), zi + 300)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), gridspec_kw=dict(width_ratios=[1.3, 1.3, 1]))
ax[0].plot(z_axis[sl]*1e3, recon_rf[sl, li], lw=0.7, label="reconstructed RF"); ax[0].plot(z_axis[sl]*1e3, envelope[sl, li], lw=2, label="envelope")
ax[0].set(title="envelope detection (centre line)", xlabel="depth (mm)"); ax[0].legend()
ax[1].plot(z_axis[sl]*1e3, 20*np.log10(np.maximum(envelope[sl, li]/envelope.max(), 1e-12)))
ax[1].set(title="log compression (dB re frame peak)", xlabel="depth (mm)", ylim=(-DR-10, 3))
ax[2].imshow(bmode, aspect="equal", cmap="gray", vmin=0, vmax=1, extent=bf.extent_mm); ax[2].set(title="T7 runner 7 output: bmode", xlabel="x (mm)", ylabel="z (mm)")
plt.tight_layout(); plt.show()

---
# T8 Post-Image Processing, runner 8: scan conversion and despeckle
**Receives:** `bmode`, scan-line positions, `z_axis`. **Does:** (1) scan conversion: a straight rectangular resample for a linear array (no fan), square pixels (`scan_convert`); (2) despeckling with the guided filter chosen in the Post-Image Processing study, radius 4, eps 0.001 (`postprocess`).
**Outputs:** `image` (H, W) and **`final`** (H, W) in [0, 1], with axes `x_img`, `z_img` in metres.

In [ ]:
image, x_img, z_img = scan_convert(bmode, x_lines, z_axis)
final = postprocess(image, method=DESPECKLE, r=post.guided_r, eps=post.guided_eps)
extent = [x_img[0]*1e3, x_img[-1]*1e3, z_img[-1]*1e3, z_img[0]*1e3]
print("image:", image.shape, f"| pixel {(x_img[1]-x_img[0])*1e3:.3f} mm | final:", final.shape)
print(f"speckle index: before {speckle_index(to_uint8(image)):.3f}, after {speckle_index(to_uint8(final)):.3f}  (lower = smoother)")
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for a, im, t in zip(ax, (image, final), ("scan-converted", f"post-processed ({DESPECKLE})")):
    a.imshow(im, cmap="gray", vmin=0, vmax=1, extent=extent); a.set(title=t, xlabel="x (mm)", ylabel="z (mm)")
plt.tight_layout(); plt.show()

### Runner 8: seeing the filter's effect
The change is small on the full image, so it is shown three ways: zoom, difference map (x10), and brightness along one column.

In [ ]:
h, w = image.shape; r0, r1, c0, c1 = h//3, h//3 + h//4, w//3, w//3 + w//3
diff = np.clip(0.5 + 10*(final - image), 0, 1)
fig, ax = plt.subplots(1, 4, figsize=(17, 4.2), gridspec_kw=dict(width_ratios=[1, 1, 1, 1.3]))
ze = [x_img[c0]*1e3, x_img[c1-1]*1e3, z_img[r1-1]*1e3, z_img[r0]*1e3]
ax[0].imshow(image[r0:r1, c0:c1], cmap="gray", vmin=0, vmax=1, extent=ze); ax[0].set_title("zoom: before")
ax[1].imshow(final[r0:r1, c0:c1], cmap="gray", vmin=0, vmax=1, extent=ze); ax[1].set_title("zoom: after")
ax[2].imshow(diff, cmap="gray", vmin=0, vmax=1, extent=extent); ax[2].set_title("(after - before) x 10, grey = 0")
col = w // 2
ax[3].plot(z_img*1e3, image[:, col], lw=0.8, label="before"); ax[3].plot(z_img*1e3, final[:, col], lw=1.5, label="after")
ax[3].set(title="one column (x = centre)", xlabel="depth (mm)"); ax[3].legend()
plt.tight_layout(); plt.show()

---
# FINAL IMAGE

In [ ]:
plt.figure(figsize=(5, 6))
plt.imshow(final, cmap="gray", vmin=0, vmax=1, extent=extent)
plt.colorbar(label=f"brightness (0 = -{DR:g} dB, 1 = 0 dB)", fraction=0.05)
plt.xlabel("lateral x (mm)"); plt.ylabel("depth z (mm)"); plt.title(f"Final B-mode: {SOURCE}")
plt.savefig("final_image_demo.png", dpi=150, bbox_inches="tight"); plt.show()
np.save("final_image_demo.npy", final)
print("saved:", Path("final_image_demo.png").resolve(), "and final_image_demo.npy")

### Every team's output side by side (this run)\nOne panel per team, left to right in pipeline order.

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(18, 9)); ax = ax.ravel()
ax[0].scatter(element_x*1e3, np.zeros_like(element_x), marker="s", s=10); ax[0].set_yticks([]); ax[0].set(title="T1 probe: element positions", xlabel="x (mm)")
ax[1].plot(tx["element_x_m"]*1e3, tx["tx_delays_s"]*1e6); ax[1].set(title=f"T2 transmit delays (focus {TX_FOCUS_MM:g} mm)", xlabel="x (mm)", ylabel="us")
ax[2].imshow(disp.T, aspect="auto", cmap="gray", vmin=-lim, vmax=lim, extent=[0.5, rc.n_elements+0.5, relay.t_axis[-1]*1e6, relay.t_axis[0]*1e6]); ax[2].set(title="T3 received_pressure", xlabel="element", ylabel="us")
ax[3].scatter(xyz[order, 0]*1e3, xyz[order, 2]*1e3, c=amp[order], s=2, cmap="gray_r", vmin=0, vmax=np.percentile(amp, 99)); ax[3].invert_yaxis(); ax[3].set(title="T4 scatterers", xlabel="x (mm)", ylabel="z (mm)")
rl = np.percentile(np.abs(beamformed_rf), 99.5); ax[4].imshow(beamformed_rf, aspect="auto", cmap="gray", vmin=-rl, vmax=rl, extent=bf.extent_mm); ax[4].set(title="T5 beamformed_rf", xlabel="x (mm)", ylabel="z (mm)")
rl = np.percentile(np.abs(recon_rf), 99.5); ax[5].imshow(recon_rf, aspect="auto", cmap="gray", vmin=-rl, vmax=rl, extent=bf.extent_mm); ax[5].set(title="T6 recon_rf", xlabel="x (mm)", ylabel="z (mm)")
ax[6].imshow(bmode, aspect="auto", cmap="gray", vmin=0, vmax=1, extent=bf.extent_mm); ax[6].set(title="T7 bmode", xlabel="x (mm)", ylabel="z (mm)")
ax[7].imshow(final, cmap="gray", vmin=0, vmax=1, extent=extent); ax[7].set(title="T8 final image", xlabel="x (mm)", ylabel="z (mm)")
plt.tight_layout(); plt.show()

### Handover summary: what each team produced in this run

In [ ]:
rows = [("T1", "element_x", element_x, "m"), ("T2", "tx_delays_s", tx["tx_delays_s"], "s"), ("T2", "tx_weights", tx["tx_weights"], "0..1"), ("T4 run 1", "scatterer_xyz", relay.scatterers.xyz, "m"), ("T3 run 2", "incident_pressure", relay.incident, "a.u., N x T"),
        ("T4 run 3", "reflected_pressure", relay.reflected, "a.u., N x T"), ("T3 run 4", "received_pressure", relay.received_pressure, "a.u., elements x T"),
        ("T3 run 4", "t_axis", relay.t_axis, "s"), ("T5 run 5", "beamformed_rf", beamformed_rf, "a.u., depth x lines"), ("T5 run 5", "z_axis", z_axis, "m"),
        ("T5 run 5", "x_lines", x_lines, "m"), ("T6 run 6", "recon_rf", recon_rf, "a.u., depth x lines"), ("T7 run 7", "bmode", bmode, f"0..1 ({DR:g} dB)"), ("T8 run 8", "image", image, "0..1, px"), ("T8 run 8", "final", final, "0..1, px")]
print(f"{'team/run':9s} {'output':20s} {'shape':16s} units")
for r, n, a, u in rows:
    print(f"{r:9s} {n:20s} {str(a.shape):16s} {u}")

---
## Use your own phantom later
Use the **Upload** cell near the top, or set `SOURCE` / `MAPPING` in the settings cell. From a terminal: `python run_pipeline.py --upload` opens a file dialog, and `python run_pipeline.py` asks what to run.
For the mapping file format see `teams/T4_Tissue_Interaction/sample_data/phantom_abdomen_mapping.json` and `examples/datasets/`.

---
## Bonus: the same chain in ONE call, on the nine-wire phantom (about 5 s)
`run_pipeline(...)` does runners 1 to 8 at once. This is what `python run_pipeline.py --source points` runs.

In [ ]:
t0 = time.time()
res = run_pipeline(demo_points(), RelayConfig(also_ti_channel=False), verbose=False)
print(f"run_pipeline: {time.time()-t0:.1f} s | final {res.final.shape} | scatterers {len(res.full.relay.scatterers.xyz)}")
plt.figure(figsize=(4, 5)); plt.imshow(res.final, cmap="gray", vmin=0, vmax=1, extent=res.extent_mm)
plt.xlabel("x (mm)"); plt.ylabel("z (mm)"); plt.title("nine wires, one call"); plt.show()